# NLP Experiment 1 – Text preprocessing and frequent terms (AG News)
**Runtime:** Python 3 (CPU). Runtime → Run all.

Pipeline: **lowercase → remove punctuation / special characters → tokenize → remove stop words → stem / lemmatize → count frequent terms**.
Dataset: **AG News** (news headlines + descriptions, 4 classes: World, Sports, Business, Sci/Tech).

In [ ]:
!pip install -q datasets nltk

In [ ]:
import re
from collections import Counter

import nltk
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

## Load a set of news articles (5,000 for speed)

In [ ]:
dataset = load_dataset("sh0416/ag_news", split="train")
df = pd.DataFrame(dataset).head(5000)

df["text"] = df["title"] + ". " + df["description"]
print("Articles:", len(df))
print(df["text"][0])

## (d) Normalize: convert to lowercase

In [ ]:
sample = df["text"][0]

lower = sample.lower()
print(lower)

## (a) Remove punctuation and special characters (keep only letters and spaces)

In [ ]:
clean = re.sub(r"[^a-z\s]", " ", lower)     # anything that is not a-z or whitespace -> space
clean = re.sub(r"\s+", " ", clean).strip()   # collapse repeated spaces
print(clean)

## (b) Tokenization: sentences, words, characters

In [ ]:
print("Sentence tokens:", sent_tokenize(sample))
print("Word tokens:", word_tokenize(clean))
print("Character tokens:", list(clean[:30]))

## (a) Remove stop words (very common words like *the, is, of*)

In [ ]:
stop_words = set(stopwords.words("english"))

tokens = word_tokenize(clean)
filtered = [w for w in tokens if w not in stop_words]

print("Before:", tokens)
print("After :", filtered)

## (c) Stemming (chop suffixes) vs Lemmatization (dictionary root word)

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("Stemmed   :", [stemmer.stem(w) for w in filtered])
print("Lemmatized:", [lemmatizer.lemmatize(w) for w in filtered])

for w in ["studies", "running", "better", "companies"]:
    print(f"{w:10} stem: {stemmer.stem(w):8} lemma: {lemmatizer.lemmatize(w)}")

## Full pipeline on every article

In [ ]:
def preprocess(text):
    text = text.lower()                              # (d) lowercase
    text = re.sub(r"[^a-z\s]", " ", text)            # (a) punctuation / special characters
    tokens = word_tokenize(text)                      # (b) tokenize
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]   # (a) stop words
    tokens = [lemmatizer.lemmatize(w) for w in tokens]                   # (c) lemmatize
    return tokens

df["tokens"] = df["text"].apply(preprocess)
print(df["tokens"][0])

## Analyse the most frequent terms

In [ ]:
all_words = []
for tokens in df["tokens"]:
    all_words.extend(tokens)

freq = Counter(all_words)
top20 = freq.most_common(20)

print("Total tokens:", len(all_words), " Unique terms:", len(freq))
pd.DataFrame(top20, columns=["term", "frequency"])

In [ ]:
words  = [w for w, c in top20]
counts = [c for w, c in top20]

plt.figure(figsize=(12, 5))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 20 most frequent terms (AG News)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()